# RAG Diagnostics Report Template

This notebook is a guided offline report for diagnosing a RAG pipeline using datasets exported from the backend evaluation workflow. It is designed for portfolio use and assumes the RAG results already exist as local CSV or JSON files.

## Purpose
Use this notebook to validate schema, compare retrieval quality, review metric reasons, and write the final engineering conclusion without requiring a live API call.

## Data contract and assumptions

The report assumes the RAG outputs were exported from a local evaluation pipeline and then saved as JSON or CSV files. The notebook does not call the API; it only loads precomputed results.

### Checklist
- [ ] The dataset includes one row per query or test case.
- [ ] Each row includes the RAG execution metadata.
- [ ] The retrieval context is stored locally for diagnostics.
- [ ] The files are documented and reproducible.

## Endpoint schema and execution metadata

The report mirrors the backend contract and keeps the execution flags visible so the analysis can distinguish retrieval strategy from generation behavior.

### Expected fields
- `Question`
- `Actual Answer`
- `Expected Output`
- `retrieval_context`
- `search_type`
- `search_method`
- `use_hyde`
- `model_name`
- `latency_s`
- `* Score` metric columns
- `* Reason` metric columns

### Checklist
- [ ] The schema is aligned with the offline export format.
- [ ] The retrieval strategy is visible in each run.
- [ ] The report preserves qualitative reasons for inspection.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

ENDPOINT_CORE_COLUMNS = [
    'Question',
    'Actual Answer',
    'Expected Output',
]

METRIC_SCORE_COLUMNS = [
    'Correctness Score',
    'Faithfulness Score',
    'Answer Relevancy Score',
    'ContextualPrecision Score',
    'ContextualRecall Score',
    'ContextualRelevancy Score',
]

METRIC_REASON_COLUMNS = [
    'Correctness Reason',
    'Faithfulness Reason',
    'Answer Relevancy Reason',
    'ContextualPrecision Reason',
    'ContextualRecall Reason',
    'ContextualRelevancy Reason',
]


def load_table(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    if path.suffix.lower() == '.json':
        with path.open('r', encoding='utf-8') as handle:
            data = json.load(handle)
        if isinstance(data, list):
            return pd.DataFrame(data)
        return pd.DataFrame([data])
    raise ValueError(f'Unsupported file type: {path.suffix}')


def normalize_endpoint_export(df):
    if df.empty:
        return df
    rename_map = {
        'question': 'Question',
        'query': 'Question',
        'actual_answer': 'Actual Answer',
        'answer': 'Actual Answer',
        'result': 'Actual Answer',
        'expected_output': 'Expected Output',
        'reference': 'Expected Output',
        'latency': 'latency_s',
    }
    return df.rename(columns={source: target for source, target in rename_map.items() if source in df.columns})


def validate_endpoint_export(df):
    required_columns = ENDPOINT_CORE_COLUMNS + METRIC_SCORE_COLUMNS
    missing_columns = [column for column in required_columns if column not in df.columns]
    return {
        'valid': not missing_columns,
        'missing': missing_columns,
    }


def load_evaluation_runs(source_map):
    frames = []
    for run_name, file_path in source_map.items():
        frame = load_table(file_path)
        if frame.empty:
            print(f'Skipping missing or empty file for {run_name}: {file_path}')
            continue
        frame = normalize_endpoint_export(frame.copy())
        frame['run_name'] = run_name
        if '-' in run_name:
            system_label, scenario = run_name.split('-', 1)
        else:
            system_label, scenario = run_name, 'Unknown'
        frame['system_type'] = system_label
        frame['scenario'] = scenario
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def detect_metric_columns(df):
    return [column for column in df.columns if column.endswith(' Score')] if not df.empty else []


def build_summary_table(df):
    metric_columns = detect_metric_columns(df)
    if df.empty or not metric_columns:
        return pd.DataFrame()
    group_columns = ['run_name', 'system_type', 'scenario', 'search_type', 'search_method', 'use_hyde', 'model_name']
    numeric_columns = metric_columns + (['latency_s'] if 'latency_s' in df.columns else [])
    existing_group_columns = [column for column in group_columns if column in df.columns]
    return df.groupby(existing_group_columns, as_index=False)[numeric_columns].mean()


def build_wins_table(summary_df, metric_columns):
    if summary_df.empty or not metric_columns:
        return pd.DataFrame()
    rows = []
    for scenario_name in summary_df['scenario'].dropna().unique():
        subset = summary_df[summary_df['scenario'] == scenario_name]
        for metric in metric_columns:
            best_row = subset.loc[subset[metric].idxmax()]
            rows.append({
                'scenario': scenario_name,
                'metric': metric,
                'winner': best_row['run_name'] if 'run_name' in best_row else best_row['system_type'],
                'score': best_row[metric],
            })
    return pd.DataFrame(rows)


def build_reason_table(df, max_rows=8):
    reason_columns = [column for column in df.columns if column.endswith(' Reason')]
    if df.empty or not reason_columns:
        return pd.DataFrame()
    preview_columns = ['run_name', 'system_type', 'scenario', 'search_method', 'use_hyde', 'Question'] + reason_columns
    preview_columns = [column for column in preview_columns if column in df.columns]
    return df[preview_columns].head(max_rows)


def plot_metric_bars(summary_df, metric_columns):
    if summary_df.empty or not metric_columns:
        print('No data available for metric bars yet.')
        return
    melted = summary_df.melt(id_vars=['run_name', 'system_type', 'scenario'], value_vars=metric_columns, var_name='metric', value_name='score')
    g = sns.catplot(
        data=melted,
        kind='bar',
        x='metric',
        y='score',
        hue='run_name',
        col='scenario',
        palette='Set2',
        height=4.5,
        aspect=1.4,
        sharey=True,
    )
    g.set_xticklabels(rotation=45, ha='right')
    g.set_titles('{col_name}')
    g.figure.suptitle('RAG metric comparison by strategy and scenario', y=1.05)
    plt.show()


def plot_metric_distributions(df, metric_columns):
    if df.empty or not metric_columns:
        print('No data available for distribution plots yet.')
        return
    melted = df.melt(id_vars=['run_name', 'system_type', 'scenario'], value_vars=metric_columns, var_name='metric', value_name='score')
    _, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=melted, x='metric', y='score', hue='run_name', ax=ax, palette='Set3')
    ax.set_title('Score dispersion by metric')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


def plot_latency_scatter(df):
    if df.empty or 'latency_s' not in df.columns:
        print('Latency is not available in the current dataset.')
        return
    quality_columns = [column for column in detect_metric_columns(df) if column != 'latency_s']
    df_plot = df.copy()
    df_plot['quality_mean'] = df_plot[quality_columns].mean(axis=1)
    _, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=df_plot, x='latency_s', y='quality_mean', hue='run_name', style='scenario', s=90, ax=ax)
    ax.set_title('Latency vs average quality')
    ax.set_xlabel('Latency (s)')
    ax.set_ylabel('Average quality score')
    plt.tight_layout()
    plt.show()

## 1. Load and normalize evaluation outputs

Provide the exported files from your offline RAG evaluation here. The notebook will normalize the columns and keep missing files visible instead of failing silently.

### Checklist
- [ ] Every run has a clear label.
- [ ] The files are stored locally.
- [ ] The notebook can skip missing files safely.
- [ ] The output structure is stable enough for reporting.

In [ ]:
RESULT_SOURCES = {
    'RAG-VectorOnly-Expert': 'path/to/vector_only_expert.csv',
    'RAG-Hybrid-Expert': 'path/to/hybrid_expert.csv',
    'RAG-HybridHyDE-Expert': 'path/to/hybrid_hyde_expert.csv',
    'RAG-VectorOnly-Operational': 'path/to/vector_only_operational.csv',
    'RAG-Hybrid-Operational': 'path/to/hybrid_operational.csv',
    'RAG-HybridHyDE-Operational': 'path/to/hybrid_hyde_operational.csv',
}

all_results_df = load_evaluation_runs(RESULT_SOURCES)
print(f'Loaded rows: {len(all_results_df)}')
if all_results_df.empty:
    print('No local evaluation files were found yet. Replace the placeholder paths with exported datasets.')
else:
    display(all_results_df.head())

## 2. Validate schema and metric coverage

Standardize the output before comparing strategy variants. This step is where you check that the report has the fields needed for both retrieval diagnostics and generation review.

### Checklist
- [ ] Core columns exist.
- [ ] Metric columns are present.
- [ ] Latency exists or is explicitly marked as missing.
- [ ] Retrieval context is available for diagnostic metrics.

In [ ]:
schema_status = validate_endpoint_export(all_results_df)
metric_columns = detect_metric_columns(all_results_df)
summary_df = build_summary_table(all_results_df)

print(schema_status)
if not all_results_df.empty:
    display(all_results_df.head())
if not summary_df.empty:
    display(summary_df.head())
else:
    print('Summary table will appear here once the real exports are loaded.')

## 3. Retrieval vs generation metric review

Use the reason columns to separate retrieval problems from generation problems. The report should explain whether the failure comes from the retriever, the generator, or both.

### Checklist
- [ ] Retrieval reasons are available.
- [ ] Generation reasons are available.
- [ ] The narrative can quote the reasons directly.
- [ ] The trade-off is visible in the report.

In [ ]:
reason_snapshot_df = build_reason_table(all_results_df)
if reason_snapshot_df.empty:
    print('No metric reasons are available yet.')
else:
    display(reason_snapshot_df)


## 4. KPI tables

Use summary tables to compare the RAG variants side by side. The template should leave space for the final decision, but not force a winner before the data is loaded.

### Checklist
- [ ] Means are reported by strategy.
- [ ] Retrieval and generation metrics are visible together.
- [ ] Latency is shown as an operational trade-off.
- [ ] Tables are suitable for the final report.

In [ ]:
wins_df = build_wins_table(summary_df, metric_columns)
if not summary_df.empty:
    display(summary_df.round(3))
if not wins_df.empty:
    display(wins_df.round(3))
else:
    print('The KPI tables will appear here after you load real evaluation exports.')

## 5. Visual analytics

Create charts that make the retrieval trade-offs visible. This is where the report should show whether the strategy improves quality, grounding, or response time.

### Checklist
- [ ] Bar charts compare the variants.
- [ ] Boxplots show score spread.
- [ ] Latency vs quality is visible.
- [ ] The visual story supports the final recommendation.

In [ ]:
plot_metric_bars(summary_df, metric_columns)
plot_metric_distributions(all_results_df, metric_columns)
plot_latency_scatter(all_results_df)

## 6. Comparative analysis

Write the final interpretation from the evidence above. Keep the conclusion scenario-specific and make the trade-off between quality and latency explicit.

### Checklist
- [ ] Explain which strategy performs best.
- [ ] Describe whether the improvement comes from retrieval or generation.
- [ ] Mention the latency trade-off.
- [ ] Note any limitations of the evaluation design.

In [ ]:
analysis_prompts = [
    'Which RAG strategy wins the Expert scenario on correctness and faithfulness?',
    'Which strategy wins the Operational scenario on latency without sacrificing too much quality?',
    'Does HyDE improve retrieval enough to justify the added latency?',
    'Are there signs that the failure is in retrieval, generation, or both?'
]

display(Markdown('Use the prompts below to draft the narrative analysis from the tables above.'))
for prompt in analysis_prompts:
    print(f'- {prompt}')

## 7. Final conclusion

Use the evidence above to write the closing summary for the report. The final paragraph should be concise, scenario-aware, and honest about limitations.

### Checklist
- [ ] The conclusion summarizes the strongest strategy.
- [ ] The conclusion separates evidence from interpretation.
- [ ] The conclusion notes evaluation limitations.
- [ ] The conclusion ends with a practical recommendation.

### Conclusion placeholder
Replace this text with the final report summary once the real offline RAG results are loaded.